## Overview
As an automotive supplier, we are interested in understanding the broader trends in the automotive
industry. For this task, you will create a data pipeline to acquire, process, and load data from various
public sources.

## Objective & Tasks
Your objective is to create a script or a series of scripts (preferably in Python or SQL) that:
**1. Data Acquisition:** Write scripts to download the datasets from the provided public data sources

**2. Data Processing:** Clean and integrate these datasets. This should include, but not be limited to,
handling missing values, duplicates, and possible outliers.

**3. Data Transformation:** Transform the data into a format suitable for further analysis. Justify the
choices you make during this process.

**4. Data Loading:** Write a script to load the data into a hypothetical data storage system. While you
cannot actually load the data into Azure SQL Database or Databricks Delta Lake, you should
simulate the process and include the relevant commands in your script.

**5. Automation Suggestion:** Describe how you would automate this pipeline with a schedule interval
you would choose and explain why.

# Use the following data sources for this task:

U.S. Department of Transportation - National Highway Traffic Safety Administration: Vehicle
Complaints https://fueleconomy.gov/feg/ws/index.shtml


# Datacard

FuelEconomy.gov is federal government website that helps consumers make informed fuel economy choices when purchasing a vehicle and helps them achieve the best fuel economy possible from the cars they own.

FuelEconomy.gov is maintained by the U.S. Department of Energy's (DOE's) Office of Energy Efficiency and Renewable Energy with data provided by the U.S. Environmental Protection Agency (EPA). The site helps fulfill DOE and EPA's responsibility under the Energy Policy Act of 1992 to provide accurate fuel economy information to consumers.

The Find a Car vehicle table contains fuel economy information for 1984-current model year vehicles, and also for the Data description check: https://fueleconomy.gov/feg/ws/index.shtml

##Step 1: Download ZIP file

Download csv file from https://fueleconomy.gov/feg/ws/index.shtml which contains car vehicle table contains fuel economy information for 1984-current model year vehicles.

## Step 2: Create DBFS folder for RAW CSV

Created separate folder in DBFS to store the raw csv

Move the CSV to correct path

Verify the file path

In [0]:
dbutils.fs.mkdirs("/mnt/epa/raw")

In [0]:
dbutils.fs.mv(
    "dbfs:/FileStore/vehicles_f-1.csv",
    "dbfs:/mnt/epa/raw/",
    recurse=True
)

In [0]:
display(dbutils.fs.ls("dbfs:/mnt/epa/raw/"))

## Step 3: Read the data from the csv raw

In [0]:
df_bronze = spark.read.csv(
    "dbfs:/mnt/epa/raw/vehicles_f-1.csv",
    header=True,
    inferSchema=True
)

df_bronze.printSchema()

## Step 4: Add a new column Data_update_timestamp dynamically.

This column will store the current timestamp each time the pipeline runs (including incremental updates).

It automatically captures the current system timestamp when the pipeline runs. Works for full load and incremental updates.

In [0]:
from pyspark.sql.functions import current_timestamp

# Add timestamp column
df_bronze = df_bronze.withColumn("Data_update_timestamp", current_timestamp())

## Step 5: Write the DataFrame to Bronze Delta Table

Replaces all invalid characters in column names with underscores, allowing you to save the DataFrame as a Delta table without errors.



In [0]:
invalid_chars = [' ', ',', ';', '{', '}', '(', ')', '\n', '\t', '=']
def clean_col(col_name):
    for ch in invalid_chars:
        col_name = col_name.replace(ch, '_')
    return col_name

df_bronze = df_bronze.toDF(*[clean_col(c) for c in df_bronze.columns])

bronze_path = "dbfs:/mnt/epa/bronze"
df_bronze.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(bronze_path)

print(f"Bronze table saved at {bronze_path}")

## Step 6: Register the bronze table in the metastore for SQL access



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS epa_vehicle.bronze
USING DELTA
LOCATION '{bronze_path}'
""")

## Step 7: Display the bronze table

In [0]:
spark.read.table("epa_vehicle.bronze").display()